In [0]:
from pyspark.sql import functions as F

geography_df = spark.table("workspace.amazon_fullfilment_gold.dim_geography")
orders_with_geo = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .join(spark.table("workspace.amazon_fullfilment_silver.customer_silver"), "customer_id", "left")
    .join(spark.table("workspace.amazon_fullfilment_silver.address_silver").alias("address"),  "customer_id", "left")
    .join(spark.table("workspace.amazon_fullfilment_silver.order_items_silver").alias("order_items"), "order_id", "left")
    .join(geography_df, ["City", "Province"], "left") # Join on strings to get the key
)

# 1. Pre-aggregate items so we have 1 row per order
items_summarized = (spark.table("workspace.amazon_fullfilment_silver.order_items_silver")
    .groupBy("order_id")
    .agg(F.sum("quantity").alias("total_items_in_order"))
)

# 2. Join orders to Geography and our summarized items
daily_fact_df = (spark.table("workspace.amazon_fullfilment_silver.orders_silver").alias("o")
    .join(spark.table("workspace.amazon_fullfilment_silver.address_silver").alias("a"), "customer_id", "left")
    .join(items_summarized, "order_id", "left")
    .join(geography_df, ["City", "Province"], "left")
    
    # 3. Final Aggregation
    .withColumn("date_key", F.date_format("o.updated_timestamp", "yyyyMMdd").cast("int"))
    .groupBy("date_key", "geo_key")
    .agg(
        F.count("order_id").alias("total_orders"), # Now safe to use count()
        F.sum("total_items_in_order").alias("total_items"),
        F.current_timestamp().alias("_last_updated_timestamp")
    )
)

display(daily_fact_df)

# Save to Gold
daily_fact_df.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_gold.fact_daily_order_volume")